In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import asyncio
import pathlib
import time

import ipywidgets as widgets
import numpy as np
from PIL import Image

from maia_waterfall_widget import Waterfall, WaterfallShape

In [3]:
im = Image.open(pathlib.Path("bindings") / "src" / "waterfall.jpg")
# want to reproduce the image, so that means it is already in "dB" and we need to convert to "linear" power
# scale [0, 255] range to [-80, -12.3] dB and convert to linear
specgram_linear = (10**((67.7 * (np.asarray(im, dtype=np.float64) / 255) - 80) / 10)).astype(dtype=np.float32)

In [4]:
w_shape = WaterfallShape(time=512, freq=specgram_linear.shape[1])
w = Waterfall(w_shape, freq_samprate_hz=(915e6, 960e6), waterfall_update_rate_hz = 29.296875, layout=widgets.Layout(height='33vh'))
w

In [5]:
spectrum_visible = widgets.Checkbox(
    value=False,
    description="Show spectrum",
)
widgets.jslink((spectrum_visible, 'value'), (w, 'spectrum_visible'))
waterfall_visible = widgets.Checkbox(
    value=True,
    description="Show waterfall",
)
widgets.jslink((waterfall_visible, 'value'), (w, 'waterfall_visible'))
cmap_select = widgets.Dropdown(
    options=['viridis', 'turbo', 'inferno'],
    value='turbo',
    description="Colormap:",
)
widgets.link((cmap_select, 'value'), (w, 'colormap'))
min_max_db = widgets.IntRangeSlider(
    value=[-75, -5],
    min=-120,
    max=0,
    step=1,
    continuous_update=False,
    description="Scale (dB):",
)
widgets.link((min_max_db, 'value'), (w, 'waterfall_min_db'), transform=[
    lambda src: src[0],
    lambda tgt: (tgt, min_max_db.value[1]),
])
widgets.link((min_max_db, 'value'), (w, 'waterfall_max_db'), transform=[
    lambda src: src[1],
    lambda tgt: (min_max_db.value[0], tgt),
])
widgets.HBox([waterfall_visible, spectrum_visible, cmap_select, min_max_db])

In [6]:
async def put_spectra():
    while True:
        for spectrum_linear in specgram_linear:
            w.put_spectrum(spectrum_linear)
            await asyncio.sleep(.034)
## can optionally run the task without blocking, but then you need to explicitly .cancel() it
# put_spectra_task = asyncio.create_task(put_spectra())
# put_spectra_task

In [7]:
try:
    async with asyncio.TaskGroup() as group:
        group.create_task(put_spectra())
except asyncio.CancelledError:
    pass